# Unity Catalog and Lineage

The original three cells use v1. The following v2 registration and verification cells correspond to the successful checks captured in the November 7 screenshots. Re-export this notebook from Databricks to capture the saved outputs.

In [ ]:

schema = "dsai3202_databricks.proj_flight60106354"

processed_path = (
    "abfss://processed@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/v1/clean_parquet/"
)

curated_path = (
    "abfss://curated@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/v1/model_ready/"
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema}.flights_clean_2025
USING PARQUET
LOCATION '{processed_path}'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema}.flights_features_2025
USING PARQUET
LOCATION '{curated_path}'
""")

print("PASS: Both external tables registered")


In [ ]:

schema = "dsai3202_databricks.proj_flight60106354"

tables = spark.sql(f"SHOW TABLES IN {schema}")
display(tables)

clean_count = spark.table(
    f"{schema}.flights_clean_2025"
).count()

feature_count = spark.table(
    f"{schema}.flights_features_2025"
).count()

assert clean_count == 6879484
assert feature_count == 6879484

print("Clean table:", clean_count)
print("Feature table:", feature_count)
print("PASS: Unity Catalog tables verified")


In [ ]:

from pyspark.sql import functions as F

schema = "dsai3202_databricks.proj_flight60106354"

real_eda = spark.table(
    f"{schema}.flights_features_2025"
)

total = real_eda.count()
delayed = real_eda.filter(F.col("ArrDel15") == 1).count()
on_time = total - delayed

print("REAL 2025 FLIGHT STATISTICS")
print("Total valid flights:", total)
print("Delayed flights:", delayed)
print("On-time flights:", on_time)
print(f"Delay rate: {100 * delayed / total:.2f}%")

display(
    real_eda.groupBy("Reporting_Airline")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("DelayRate")
    )
    .orderBy(F.desc("TotalFlights"))
)


In [ ]:
# Register the independent production v2 outputs, leaving v1 unchanged.
schema = "dsai3202_databricks.proj_flight60106354"
processed_v2 = "abfss://processed@stflight60106354.dfs.core.windows.net/flight_delays/2025/v2/clean_parquet/"
curated_v2 = "abfss://curated@stflight60106354.dfs.core.windows.net/flight_delays/2025/v2/model_ready/"
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.flights_clean_2025_v2 USING PARQUET LOCATION '{processed_v2}'")
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.flights_features_2025_v2 USING PARQUET LOCATION '{curated_v2}'")
print("PASS: Production v2 tables registered")

In [ ]:
# Verify production v2 registration and row counts.
schema = "dsai3202_databricks.proj_flight60106354"
clean_v2 = spark.table(f"{schema}.flights_clean_2025_v2")
features_v2 = spark.table(f"{schema}.flights_features_2025_v2")
clean_count = clean_v2.count()
feature_count = features_v2.count()
assert clean_count == 6879484
assert feature_count == 6879484
assert "ArrDelay" not in features_v2.columns
print("Production clean records:", clean_count)
print("Production feature records:", feature_count)
print("PASS: Production v2 catalog tables verified")